In [1]:
import pickle as pkl
import numpy as np
from torch.utils.data import Dataset, DataLoader
import pandas as pd

/home/berk/.local/lib/python3.8/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
with open ('/home/berk/VS_Projects/simglucose/online-dt/data/hopper-medium-v2.pkl', 'rb') as handle:
    orj_data = pkl.load(handle)

In [7]:
print(type(orj_data))
row=len(orj_data)
column=len(orj_data[0])
print(f'Rows:{row}, Column:{column}')

<class 'list'>
Rows:2186, Column:5


In [19]:
first_row = orj_data[0]
actions = first_row["actions"]
print(np.shape(actions))

(470, 3)


In [11]:
print(actions)

[[ 0.934155   -0.71524495 -0.10588102]
 [ 0.35850546 -0.71467435 -0.15425174]
 [ 0.17269637 -0.3426832  -0.52400124]
 ...
 [-0.58971727 -0.57238317  0.8646174 ]
 [-0.27115864  0.1876209  -0.6353513 ]
 [ 0.43704683 -0.4414192   0.505635  ]]


In [12]:
own_data = pd.read_csv("/home/berk/VS_Projects/simglucose/examples/T1DatasetAnalysis/PPO/seed0/adolescent#001.csv")

In [15]:
type(own_data)
own_data.describe()
own_data.head()

,no,Action,CHO,reward,BG,CGM,RI,LBGI,HBGI,MRI
0,0,0.000008,0.0,1.0,155.583498,164.647835,3.668147,0.0,3.668147,0.595842
1,1,0.000000,0.0,1.0,154.942206,163.893527,3.574990,0.0,3.574990,0.553341
2,2,0.000000,0.0,1.0,154.394367,163.081224,3.496061,0.0,3.496061,0.518141
3,3,0.000000,0.0,1.0,153.921872,162.242153,3.428477,0.0,3.428477,0.488613
4,4,0.000000,0.0,1.0,153.511040,161.412521,3.370085,0.0,3.370085,0.463569


In [20]:
own_data_value = own_data.values
print(own_data_value)

[[0.00000000e+00 8.33333300e-06 0.00000000e+00 ... 0.00000000e+00
  3.66814722e+00 5.95842080e-01]
 [1.00000000e+00 0.00000000e+00 0.00000000e+00 ... 0.00000000e+00
  3.57498975e+00 5.53340630e-01]
 [2.00000000e+00 0.00000000e+00 0.00000000e+00 ... 0.00000000e+00
  3.49606116e+00 5.18141230e-01]
 ...
 [4.79700000e+03 0.00000000e+00 0.00000000e+00 ... 0.00000000e+00
  4.73514355e+00 1.14557648e+00]
 [4.79800000e+03 0.00000000e+00 0.00000000e+00 ... 0.00000000e+00
  4.43215642e+00 9.78920340e-01]
 [4.79900000e+03 0.00000000e+00 0.00000000e+00 ... 0.00000000e+00
  4.13666959e+00 8.23986290e-01]]


In [27]:
actions_own_data = own_data['Action'].values
states_own_data = own_data['BG'].values
reward_own_data = own_data['reward'].values
terminals_own_data = np.full(len(actions_own_data), False)
next_state_own_data = own_data['BG'].values[1:]


In [28]:
next_state_own_data = np.append(next_state_own_data, next_state_own_data[-1])

In [29]:
data_dict = {'observations': states_own_data, 'next_observations':next_state_own_data, 
             'actions': actions_own_data, 'rewards': reward_own_data, 'terminals': terminals_own_data}

In [30]:
final_data = []

In [60]:
seq_length = 200

for idx in range(0,len(states_own_data),seq_length):
    data_dict = {'observations': np.array([states_own_data[idx:seq_length+idx]]), 'next_observations':np.array([next_state_own_data[idx:seq_length+idx]]), 
                'actions': np.array([actions_own_data[idx:seq_length+idx]]), 'rewards': np.array([reward_own_data[idx:seq_length+idx]]), 
                'terminals': np.array([terminals_own_data[idx:seq_length+idx]])}
    final_data.append(data_dict)
    data_dict = {}

In [65]:
for i in range(len(final_data)):
    final_data[i]['observations']=final_data[i]['observations'].squeeze()
    final_data[i]['next_observations']=final_data[i]['next_observations'].squeeze()
    final_data[i]['rewards']=final_data[i]['rewards'].squeeze()
    final_data[i]['actions']=final_data[i]['actions'].squeeze()
    final_data[i]['terminals']=final_data[i]['terminals'].squeeze()

In [67]:
# Save the dictionary to a pickle file
with open('./PPO_traj_200_step.pkl', 'wb') as file:
    pkl.dump(final_data, file)

In [66]:
np.shape((final_data[24]['observations']))

(200,)